# COMP0189: Applied Artificial Intelligence
## Week 9 (Clustering)


### 🎯 Objectives
1. To learn how to apply different clustering approaches (K-Means, Hierarchical clustering, Spectral clustering) to an image dataset (MNIST)
2. To learn how to apply Gaussian Mixture Models (GMM) to cluster voxels corresponding to different brain tissue types (image segmentation)
3. To learn how to quantify clustering results


### Acknowledgements
- Many thanks to Prof. John Ashburner from the Wellcome Center for Human Neuroimaging (UCL) for kindly providing the brain imaging data.
- https://scikit-learn.org/stable/
- https://en.wikipedia.org/wiki/MNIST_database
- https://brain-development.org/ixi-dataset/



In [ ]:
%pip install scikit-learn==1.7.2 matplotlib==3.10.8 pandas==2.3.3 tensorflow==2.20.0 scipy==1.17.1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score
)
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs

from collections import defaultdict
from time import time

import tensorflow as tf
from tensorflow.keras import layers, Model

from scipy.cluster.hierarchy import dendrogram, linkage

# Part 1: MNIST dataset

In this part we will apply Principal Component Analysis (PCA) to the MNIST dataset and use K-Means to cluster the digits.

https://scikit-learn.org/stable/modules/clustering.html#k-means

### Task 1: Load MNIST data and assemble it in two matrices X (images) and y (labels)

In [ ]:
MNIST = np.load("mnist.npz")
for k in MNIST.files:
    print(k)

In [ ]:
MNIST["X"].shape, MNIST["y"].shape

In [ ]:
mnist_X = MNIST["X"]
mnist_y = MNIST["y"]

### Task 2: Visualise the data for better understanding

In [ ]:
def make_img_grid(images, n_cols=10):
    """Helper function for arranging images into a grid"""
    cols = []
    gap = len(images) % n_cols
    if gap > 0:
        # add padding if needed
        images = np.concatenate(
            (images, np.zeros((n_cols - gap,) + images[0].shape)), 0
        )
    for n in range(n_cols):
        cols.append(np.concatenate(images[np.arange(n, len(images), step=n_cols)]))
    return np.concatenate(cols, -1)


plt.figure(figsize=(14, 7))
plt.imshow(make_img_grid(mnist_X[:300], n_cols=30), cmap="binary_r")

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(12, 12))
plt.gray()

# loop through subplots and add mnist images
for i, ax in enumerate(axs.flat):
    ax.imshow(mnist_X[i])
    ax.axis("off")
    ax.set_title("Number {}".format(mnist_y[i]))

# display the figure
plt.show()

In order to apply PCA to the MNIST data we need to reshape the original MNIST data, from

    mnist_images.shape == [60000, 28, 28]

into a 2d array (matrix)

    X_mnist.shape == [60000, 784]


In [ ]:
# Reshaping
mnist_X_original = mnist_X.copy()
mnist_X = mnist_X.reshape(len(mnist_X), -1)
print(mnist_X.shape)

### Task 3: Apply PCA to the MNIST data

In [ ]:
# Your code here...

### Task 4: Plot the MNIST data projected onto the first two principal components (using different colours for the different digits). Use the labels to colour the examples

In [ ]:
# Your code here...

### Task 5: Plot the explained variance per component

In [ ]:
# Determine explained variance using explained_variance_ratio_ attribute
# explained_variance_ratio_
#   Percentage of variance explained by each of the selected components.
#   If n_components is not set then all components are stored and the sum of the ratios is equal to 1.0.

exp_var_pca = pca.# Your code here...

# Cumulative sum of eigenvalues will be used to create step plot
# for visualizing the variance explained by each principal component.
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

# Create the plot
plt.bar(
    range(0, len(exp_var_pca)),
    exp_var_pca,
    alpha=0.5,
    align="center",
    label="Individual explained variance",
)
plt.step(
    range(0, len(cum_sum_eigenvalues)),
    cum_sum_eigenvalues,
    where="mid",
    label="Cumulative explained variance",
)
plt.ylabel("Explained variance ratio")
plt.xlabel("Principal component index")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

### Manual way of gettting explained variance

In [ ]:
def compute_PCA_parameters(X, M):
    """
    This function computes the first M prinicpal components of a
    dataset X. It returns the mean of the data, the projection matrix,
    and the associated singular values.

    While you can compute this however you want, `np.linalg.svd` is
    highly recommended. Please look at its documentation to choose
    its arguments appropriately, and on how to interpret its return values.

    INPUT:
    X    : (N, D) matrix; each row is a D-dimensional data point
    M    : integer, <= D (number of principal components to return)

    OUTPUTS:
    x_bar  : (D,) vector, with the mean of the data
    W      : (D, M) semi-orthogonal matrix of projections
    s      : (D,) vector of singular values
    """
    N, D = X.shape

    # center the data
    x_bar = np.mean(X, 0) # compute mean across samples
    X_bar = X - x_bar # subtract mean

    # perform singular value decomposition on centered dataset
    u, s, vh = np.linalg.svd(X_bar, full_matrices=False)

    # extract principal components and singular values
    W = vh.T[:, :M]
    return x_bar, W, s

In [ ]:
mnist_mean, W_mnist, s_mnist = # Your code here...

N, data_dim = mnist_X.shape
plt.figure(figsize=(12, 4))
plt.plot(np.arange(data_dim) + 1, s_mnist**2 / N, ".-")
plt.xlabel("Component")
plt.ylabel("Explained variance")

### Task 6: Apply KMeans to cluster the MNIST data

- Preprocess the MNIST dataset with PCA to compress it down to a 2-dimensional feature space before applying the K-Means

- Try K-Means with different numbers of clusters and use the Silhouette Coefficient to choose the optimal number of cluster

- Try a different number of PCs and quantify the results using different metrics

Discussion: Does the Silhouette Coefficient chooses the right number of clusters?

In [ ]:
pca = PCA(n_components=2)
mnist_X_pca = pca.fit_transform(mnist_X)
n_digits = len(np.unique(mnist_y))
print(n_digits)

In [ ]:
%%time

kmeans = # Your code here...
kmeans.fit...# Your code here...

In [ ]:
# Generating the sample data from make_blobs
# This particular setting has one distinct cluster and 3 clusters placed close
# together.

X = # Your code here...
y = # Your code here...
range_n_clusters = [2, 5,10]

for n_clusters in range_n_clusters:
    # Create a subplot with 1 row and 2 columns
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # The 1st subplot is the silhouette plot
    # The silhouette coefficient can range from -1, 1 but in this example all
    # lie within [-0.1, 1]
    ax1.set_xlim([-0.1, 1])
    # The (n_clusters+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    ax1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

    # Initialize the clusterer with n_clusters value and a random generator
    # seed of 10 for reproducibility.
    clusterer = KMeans(n_clusters=n_clusters,  n_init=10 , random_state=10)
    cluster_labels = clusterer.fit_predict(X)

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed
    # clusters
    silhouette_avg = silhouette_score(X, cluster_labels)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to
        # cluster i, and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax1.set_title("The silhouette plot for the various clusters.")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")

    # The vertical line for average silhouette score of all the values
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

    ax1.set_yticks([])  # Clear the yaxis labels / ticks
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    # 2nd Plot showing the actual clusters formed
    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(
        X[:, 0], X[:, 1], marker=".", s=30, lw=0, alpha=0.7, c=colors, edgecolor="k"
    )

    # Labeling the clusters
    centers = clusterer.cluster_centers_
    # Draw white circles at cluster centers
    ax2.scatter(
        centers[:, 0],
        centers[:, 1],
        marker="o",
        c="white",
        alpha=1,
        s=200,
        edgecolor="k",
    )

    for i, c in enumerate(centers):
        ax2.scatter(c[0], c[1], marker="$%d$" % i, alpha=1, s=50, edgecolor="k")

    ax2.set_title("The visualization of the clustered data.")
    ax2.set_xlabel("Feature space for the 1st feature")
    ax2.set_ylabel("Feature space for the 2nd feature")

    plt.suptitle(
        "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d"
        % n_clusters,
        fontsize=14,
        fontweight="bold",
    )

plt.show()

### Task 7: Repleace the K-means Clustering by Agglomerative Clustering and Spectral Clustering
- quantify the results using different metrics



In [ ]:
pca = PCA(n_components=16)
mnist_X_pca = # Your code here...

In [ ]:
# Randomly sample a subset because some clustering methods are expensive
np.random.seed(42)
subset_size = 5000
indices = np.random.choice(mnist_X_pca.shape[0], subset_size, replace=False)

X = mnist_X_pca[indices]
y = mnist_y[indices]

print(X.shape, y.shape)

In [ ]:
def clustering_scores(X, cluster_labels, y_true):
    scores = {
        "silhouette": silhouette_score(X, cluster_labels),
        "ARI": adjusted_rand_score(y_true, cluster_labels),
        "NMI": normalized_mutual_info_score(y_true, cluster_labels),
        "homogeneity": homogeneity_score(y_true, cluster_labels),
        "completeness": completeness_score(y_true, cluster_labels),
    }
    return scores

def print_scores(name, scores):
    print(f"\n{name}")
    for k, v in scores.items():
        print(f"{k:>12s}: {v:.4f}")

In [ ]:
n_clusters = 10

# K-means
# Your code here...

# Agglomerative clustering
# Your code here...

# Spectral clustering
# Your code here...

print_scores("K-means on PCA features", kmeans_scores)
print_scores("Agglomerative on PCA features", agg_scores)
print_scores("Spectral clustering on PCA features", spectral_scores)

**Visualising hierarchical clustering with a dendrogram**

Hierarchical clustering builds a tree structure (a **dendrogram**) showing how clusters are progressively merged.

Each leaf represents a sample, and branches indicate when clusters are combined based on distance.  
Because MNIST contains many samples, we visualise the dendrogram on a **small subset** of the dataset.

This helps illustrate how agglomerative clustering forms clusters step by step.

In [ ]:
# use a subset for better readability
subset_size = 200
X_subset = X[:subset_size]

# compute hierarchical clustering linkage
Z = linkage(X_subset, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="level", p=5)
plt.title("Hierarchical Clustering Dendrogram (MNIST subset)")
plt.xlabel("Sample index")
plt.ylabel("Distance")
plt.show()

**Visualising clustering results in PCA space**

The data are plotted using the first two principal components (PC1 and PC2). Each point represents a sample and is coloured according to the cluster label assigned by each algorithm, allowing comparison of how K-means, Agglomerative, and Spectral clustering partition the data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X[:, 0], X[:, 1], c=kmeans_labels, s=5, cmap="tab10")
axes[0].set_title("K-means clusters")

axes[1].scatter(X[:, 0], X[:, 1], c=agg_labels, s=5, cmap="tab10")
axes[1].set_title("Agglomerative clusters")

axes[2].scatter(X[:, 0], X[:, 1], c=spectral_labels, s=5, cmap="tab10")
axes[2].set_title("Spectral clusters")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

We cannot easily visualise the data in the 16-dimensional PCA space. To obtain an interpretable 2D representation, we apply **t-SNE**, a nonlinear dimensionality reduction method that preserves local neighbourhood structure. This allows us to visualise how samples group together in a low-dimensional embedding.

In [ ]:
# Compute t-SNE embedding of the 16D PCA features
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X)

# Plot true labels + clustering assignments on the t-SNE map
fig, axes = plt.subplots(1, 4, figsize=(28, 5))

# True digit labels
sc0 = axes[0].scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=y, s=6, cmap="tab10", vmin=0, vmax=9
)
axes[0].set_title("True digit labels")

# K-means
sc1 = axes[1].scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=kmeans_labels, s=6, cmap="tab10", vmin=0, vmax=9
)
axes[1].set_title("K-means clusters")

# Agglomerative
sc2 = axes[2].scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=agg_labels, s=6, cmap="tab10", vmin=0, vmax=9
)
axes[2].set_title("Agglomerative clusters")

# Spectral
sc3 = axes[3].scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=spectral_labels, s=6, cmap="tab10", vmin=0, vmax=9
)
axes[3].set_title("Spectral clusters")

for ax in axes:
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

# Leave space on the right for the colorbar
fig.subplots_adjust(right=0.90)

# Shared colorbar outside the plots
cbar = fig.colorbar(sc0, ax=axes, location="right")
cbar.set_label("Digit / cluster label")

plt.show()

### Discussion
- Do clusters with high silhouette scores correspond to meaningful digit groups?

- Which clustering algorithm performed best?

- How does the dimensionality reduction step (PCA --> t-SNE) affect how clusters appear visually?

- Are some digits consistently confused or grouped together?


### Task 8: Apply KMeans to cluster the MNIST data using non-linear embeddings
- Use an Autoencoder and a Variational Autoencoder to extract embeddings from the MNIST dataset before applying K‑Means.
- Use t‑SNE to visualize the embeddings, colouring points according to the digit labels.
- Try K‑Means with different numbers of clusters and quantify results using multiple metrics

In [ ]:
# use flattened MNIST for AE/VAE
np.random.seed(42)

# scale original 0-255 values
if mnist_X.max() > 1.0:
    mnist_X = mnist_X / 255.0

# if images are 28x28, flatten them
if mnist_X.ndim == 3:
    mnist_X = mnist_X.reshape(len(mnist_X), -1)
elif mnist_X.ndim == 4:
    mnist_X = mnist_X.reshape(len(mnist_X), -1)

print(mnist_X.shape)  # should be (N, 784)

X_train, X_test, y_train, y_test = train_test_split(
    mnist_X, mnist_y, test_size=0.8, random_state=42, stratify=mnist_y
)

**Step 1: Autoencoder**

In [ ]:
latent_dim = 16

inputs = layers.Input(shape=(784,))
x = layers.Dense(256, activation="relu")(inputs)
x = layers.Dense(128, activation="relu")(x)
latent = layers.Dense(latent_dim, name="latent_vector")(x)

x = layers.Dense(128, activation="relu")(latent)
x = layers.Dense(256, activation="relu")(x)
outputs = layers.Dense(784, activation="sigmoid")(x)

autoencoder = Model(inputs, outputs)
encoder = Model(inputs, latent)

autoencoder.compile(optimizer="adam", loss="mse")

history_ae = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_test, X_test),
    epochs=20,
    batch_size=256,
    verbose=1
)

Look at sample predictions (AE reconstructions of MNIST digits)

In [ ]:
n_examples = 10
test_samples = X_test[:n_examples]

reconstructions = autoencoder.predict..# Your code here...

fig, axes = plt.subplots(2, n_examples, figsize=(15, 4))

for i in range(n_examples):
    axes[0, i].imshow(test_samples[i].reshape(28, 28), cmap="gray")
    axes[0, i].set_title(f"{y_test[i]}")
    axes[0, i].axis("off")

    axes[1, i].imshow(reconstructions[i].reshape(28, 28), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Recon", fontsize=12)

plt.tight_layout()
plt.show()

Get AE embeddings and apply K-means clustering

In [ ]:
ae_latent = encoder.predict.. # Your code here...
ae_latent_scaled = StandardScaler().fit_transform(ae_latent)

kmeans_ae = # Your code here...
ae_clusters = kmeans_ae.fit_predict..# Your code here...

# Compute clustering scores
# Your code here...

**Step 2: Variational Autoencoder (VAE)**

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

latent_dim = 16

# ----- Encoder -----
vae_inputs = layers.Input(shape=(784,), name="encoder_input")
x = layers.Dense(256, activation="relu")(vae_inputs)
x = layers.Dense(128, activation="relu")(x)

z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

vae_encoder = Model(vae_inputs, [z_mean, z_log_var, z], name="vae_encoder")

# ----- Decoder -----
latent_inputs = layers.Input(shape=(latent_dim,), name="z_sampling")
x = layers.Dense(128, activation="relu")(latent_inputs)
x = layers.Dense(256, activation="relu")(x)
vae_outputs = layers.Dense(784, activation="sigmoid")(x)

vae_decoder = Model(latent_inputs, vae_outputs, name="vae_decoder")

In [ ]:
class VAE(Model):
    def __init__(self, encoder, decoder, beta=0.001, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data, training=True)
            reconstruction = self.decoder(z, training=True)

            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.square(data - reconstruction), axis=1)
            )

            kl_loss = tf.reduce_mean(
                tf.reduce_sum(
                    -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                    axis=1
                )
            )

            total_loss = reconstruction_loss + self.beta * kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

In [ ]:
vae = VAE(vae_encoder, vae_decoder, beta=0.001)
vae.compile(optimizer=tf.keras.optimizers.Adam(1e-3))

history_vae = vae.fit(
    X_train,
    epochs=20,
    batch_size=256,
    verbose=1
)

Look at sample predictions (VAE reconstructions of MNIST digits)

In [ ]:
n_examples = 10

z_mean_test, z_log_var_test, z_test = vae_encoder.predict.. # Your code here...
vae_recon = vae_decoder.predict..# Your code here...

fig, axes = plt.subplots(2, n_examples, figsize=(15, 4))

for i in range(n_examples):
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap="gray")
    axes[0, i].set_title(f"Digit {y_test[i]}")
    axes[0, i].axis("off")

    axes[1, i].imshow(vae_recon[i].reshape(28, 28), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("VAE recon", fontsize=12)

plt.tight_layout()
plt.show()

Get VAE embeddings and apply K-means clustering

In [ ]:
z_mean_test, z_log_var_test, z_test = vae_encoder...# Your code here...

print("z_mean shape:", z_mean_test.shape)
print("overall std of z_mean:", np.std(z_mean_test))

z_mean_scaled = StandardScaler().fit_transform(z_mean_test)

kmeans_vae = # Your code here...

print("Number of unique clusters:", len(np.unique(vae_clusters)))

# Inspect clustering scores
# Your code here...

**Compare AE and VAE**: t-SNE plot of embeddings

In [ ]:
ae_latent = encoder.predict..# Your code here...
z_mean_test, _, _ = vae_encoder.predict..# Your code here...

In [ ]:
# Use a smaller subset of the TEST set because t-SNE is slow

viz_size = min(10000, len(X_test))
viz_idx = np.random.choice(len(X_test), viz_size, replace=False)

ae_latent_viz = ae_latent[viz_idx]
vae_latent_viz = z_mean_test[viz_idx]
y_viz = y_test[viz_idx]

# t-SNE
tsne_ae = TSNE(n_components=2, random_state=42, perplexity=30)
ae_tsne = tsne_ae.fit_transform(ae_latent_viz)

tsne_vae = TSNE(n_components=2, random_state=42, perplexity=30)
vae_tsne = tsne_vae.fit_transform(vae_latent_viz)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = axes[0].scatter(ae_tsne[:, 0], ae_tsne[:, 1], c=y_viz, s=6, cmap="tab10")
axes[0].set_title("t-SNE of AE latent space (test set)")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")

scatter2 = axes[1].scatter(vae_tsne[:, 0], vae_tsne[:, 1], c=y_viz, s=6, cmap="tab10")
axes[1].set_title("t-SNE of VAE latent space (test set)")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

# Create legend for digits
import matplotlib.lines as mlines
cmap = plt.cm.get_cmap("tab10")

legend_handles = [
    mlines.Line2D([], [], color=cmap(i), marker='o', linestyle='None',
                  markersize=6, label=str(i))
    for i in range(10)
]

axes[1].legend(handles=legend_handles, title="Digit",
               bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

### Interpretation of clustering results

How do these results compare to K-Means in previous tasks?

# Part 2: IXI dataset

In this part we apply use Gaussian Mixture Model (GMM) to an image segmentation task. We will use the voxel intensities of two brain images to cluster them into four different clusters that should correspond to different types of brain tissue (grey matter, white amtter, Cerebrospinal fluid (CSF) and "other").

https://scikit-learn.org/stable/modules/mixture.html#gmm

In [ ]:
%pip install nilearn==0.13.0

In [ ]:
import nibabel
import nibabel.processing
from nilearn import plotting

### Task 8: Load two brain images (proton density-weighted (PDW) and T2-weighted (T2W)) that have been previously pre-processed

In [ ]:
f1 = nibabel.load("T2.nii")
f2 = nibabel.load("PD.nii")

Resample image to have all dimensions of equal size (you don't need to worry about this).

In [ ]:
f1 =  nibabel.processing.conform(f1, out_shape=(256, 256, 256), voxel_size=(1.0, 1.0, 1.0), order=0, orientation='RAS', out_class=None)
f2 =  nibabel.processing.conform(f2, out_shape=(256, 256, 256), voxel_size=(1.0, 1.0, 1.0), order=0, orientation='RAS', out_class=None)

Get images as numpy arrays

In [ ]:
x1 = f1.get_fdata()
x2 = f2.get_fdata()

Look at images (plots are interactive!)

In [ ]:
plotting.view_img(f1, bg_img=False, black_bg=True)

In [ ]:
plotting.view_img(f2, bg_img=False, black_bg=True)

### Task 9: Vectorize the brain images and combine them to create a data matrix with 2 dimensions per voxel (PD and T2 intensities)

In [ ]:
# Flatten the 3D arrays to 2D arrays where each row is a voxel
x1_flat = x1.reshape(-1, 1) # Reshape x1 to have one column and as many rows as there are voxels
x2_flat = x2.reshape(-1, 1) # Reshape x2 similarly

# Combine the flattened arrays side by side to create a new 2D array with two columns (PD and T2 intensities)
data_matrix = np.hstack(# Your code here...)

In [ ]:
print(data_matrix.shape)

In [ ]:
# reshaping to original
original_shape = x1.shape

# Split the 2D data_matrix back into two 1D arrays
x_1_flat = data_matrix[:, 0]  # This extracts the first column (PD intensities)
x_2_flat = data_matrix[:, 1]  # This extracts the second column (T2 intensities)

# Reshape these 1D arrays back into their original 3D shape
x1_reshaped = x_1_flat.reshape(# Your code here...)
x2_reshaped = x_2_flat.reshape(# Your code here...)


In [ ]:
print(x_1_flat.shape)

In [ ]:
# check if original and un-reshape reshaped match
print(np.array_equal(x1, x1_reshaped))
print(np.array_equal(x2, x2_reshaped))

###  Task 10: Apply GMM to cluster the voxels fixing the number of cluster to 4
We expect the four classes to correspond to:
- grey matter
- white matter
- cerebrospinal fluid
- other (background)

In [ ]:
from sklearn.mixture import GaussianMixture

# Assuming data_matrix is the combined 2D array from the previous steps

# Initialize the Gaussian Mixture Model with 4 components (clusters)
gmm = # Your code here...

# Fit the model to the data and predict the cluster for each voxel
cluster_labels = gmm.fit_predict(# Your code here...)

# cluster_labels is now a 1D array with the cluster label (0, 1, 2, 3) for each voxel

In [ ]:
print(cluster_labels.shape)

In [ ]:
# For visualisation purposes add 1 to each cluster (to avoid confusion with background - although here background is a cluster so this will make no difference)
cluster_labels += 1

# Reshape cluster_labels back to the original 3D shape of the images
clustered_image = cluster_labels.reshape(# Your code here...)

# clustered_image is a 3D array with the same shape as the original images,
# where each voxel's value represents its cluster label.
print(clustered_image.shape)

In [ ]:
# Convert the clustered image to a Nifti1Image object using the correct affine matrix
clustered_img_nii = nibabel.Nifti1Image(clustered_image.astype(np.int16), affine=f1.affine)

# View the clustered image in an interactive viewer
plotting.view_img(clustered_img_nii, threshold='auto', cmap='magma', symmetric_cmap=False, vmin=0)

> **Question**: do you recognise the four classes? Do they correspond to the brain structures identified earlier?

### Task 11: Convert the class probabilities for each voxel to a 3D image

Alternative, instead of classes, based on probabilities

In [ ]:
# Predict the probabilities for each voxel belonging to each cluster

voxel_probabilities = gmm.predict_proba(# Your code here...)

# Now, voxel_probabilities is a 2D numpy array where each row represents a voxel
# and each column represents the probability of that voxel belonging to a certain cluster.

# Reshape these probabilities back to the original 3D shape for each cluster

probabilities_4d = np.zeros((x1.shape[0], x1.shape[1], x1.shape[2], gmm.n_components))

for i in range(gmm.n_components):
    # Reshape the probabilities for cluster i into the original 3D shape
    probabilities_4d[:,:,:,i] = voxel_probabilities[:,i].reshape(x1.shape)


**Question:** Which brain structure do each of these correspond to?

In [ ]:
# Convert the probability maps to NIfTI images and visualize
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster 0 Probability Map')

In [ ]:
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster 1 Probability Map')

In [ ]:
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster 2 Probability Map')

In [ ]:
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster 3 Probability Map')

### Task 11: Use three clusters only. We want to ignore the background.

We now expect the following clusters:
- grey matter
- white matter
- cerebrospinal fluid

Create and apply a mask to filter out the background. We use the same data matrix as before.

In [ ]:
# Mask for non-zero pixels
non_zero_mask = # Your code here...

# Filter out the zero-valued pixels
data_matrix_non_zero = data_matrix[non_zero_mask]

Apply GMM

In [ ]:
# Initialize the Gaussian Mixture Model
gmm = # Your code here...

cluster_labels_non_zero = gmm.fit_predict(# Your code here...)

Plot the clusters

In [ ]:
# Adjust the labels to start from 1 instead of 0 (for visualisation purposes)
cluster_labels_non_zero += 1

# Initialize an output array with zeros for the background
clustered_image = np.zeros(x1_flat.shape[0], dtype=np.int16)

# Assign the GMM cluster labels back to the non-zero pixels in the output array
clustered_image[non_zero_mask] = cluster_labels_non_zero

# Reshape cluster_labels back to the original 3D shape of the images
clustered_image_reshaped = clustered_image.reshape(# Your code here...)

# Convert the clustered image to a Nifti1Image object using the correct affine matrix
clustered_img_nii = nibabel.Nifti1Image(clustered_image_reshaped, affine=f1.affine)

# View the clustered image
plotting.view_img(clustered_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0)


### Task 12: Look at individual probabilites for each class.

In [ ]:
# Predict the probabilities for each voxel belonging to each cluster
voxel_probabilities = gmm.predict_proba(# Your code here...)

# Initialize an output 4D array for probabilities with zeros
# This array has the same width, height, and depth as the original images, and an extra dimension for each cluster
probabilities_4d = np.zeros((x1.shape[0], x1.shape[1], x1.shape[2], gmm.n_components))

# We need a way to map the probabilities back to their original positions including zeros
# First, flatten the 4D array to match the shape of the non-zero processing (x1_flat.shape[0], n_components)
probabilities_flat = probabilities_4d.reshape(-1, gmm.n_components)

# Now, assign the probabilities back to the non-zero positions in the flattened array
probabilities_flat[non_zero_mask, :] = voxel_probabilities

# Finally, reshape this flat probabilities array back to the original 4D shape
probabilities_4d = probabilities_flat.reshape(x1.shape[0], x1.shape[1], x1.shape[2], gmm.n_components)

In [ ]:
# Convert the probability maps to NIfTI images and visualize
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster {0} Probability Map')

In [ ]:
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster {1} Probability Map')

In [ ]:
prob_img_nii = nibabel.Nifti1Image(probabilities_4d[:,:,:,# Your code here...].astype(np.float32), affine=f1.affine)
plotting.view_img(prob_img_nii,
                  threshold='auto', bg_img=False, black_bg=True, cmap='magma', symmetric_cmap=False, vmin=0, title=f'Cluster {2} Probability Map')